# RG3 미성형15개 실제 저장 모델 및 재현 확인


In [ ]:
from pathlib import Path
import os,ast,nbformat,json,joblib,numpy as np
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rg3/01_preprocess_train.ipynb').exists())
n=nbformat.read(ROOT/'rg3/internal/notebooks/212_rg3_legacy_training_reference.ipynb',4)
exec(n.cells[1].source)
MODEL_DIR=WORK/'models/reduced_shortshot15';MODEL_DIR.mkdir(parents=True,exist_ok=True)
original_fit_mean=fit_mean_models
def fit_mean_models(a,y,seeds,thermal):
 if thermal:
  models=[(1/3,cls(n_estimators=100,max_depth=4,min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=2).fit(a,y)) for cls in [RandomForestClassifier,ExtraTreesClassifier]]
  models.append((1/3,SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,y)))
 else:
  models=[(.5,SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,y)),(.5,LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,y))]
 return models
previous=Path.cwd()
try:
 os.chdir(WORK)
 setup=source('145_rg3_shortshot_unlabeled_application.ipynb')
 shortns={'__name__':'__main__'};exec(setup[:setup.index('u_all=pd.read_csv')],shortns)
finally:os.chdir(previous)
tree=ast.parse(setup[:setup.index('u_all=pd.read_csv')])
defs=[node.value.args[0].value for node in tree.body if isinstance(node,ast.Expr) and isinstance(node.value,ast.Call) and isinstance(node.value.func,ast.Name) and node.value.func.id=='exec' and isinstance(node.value.args[0],ast.Constant)]
def reduce_B(s):
 return s.replace("for name,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:","for name,cls in []:").replace('for C in [.1,1.]:','for C in [1.]:').replace("['rf','et','svm0.1','svm1.0','lr']","['svm1.0','lr']")
bsource=reduce_B(defs[1]);bsource=bsource[bsource.index('def predict('):bsource.index('def threshold(')]
shortns['nb']['memo']={};exec(bsource,shortns['nb'])
def reduce_C(s):
 return s.replace('for seed in [42,7]:','for seed in [42]:').replace('for C in [.1,1.]:','for C in [1.]:').replace("m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)","m=None").replace("out[view+'_lr']=m.predict_proba(b)[:,1]","out[view+'_lr']=np.zeros(len(q))").replace("['rf','et','svm0.1','svm1.0','lr']","['rf','et','svm1.0']")
shortns['nc']=shortns['nc'][:1]
shortns['sourceC']=reduce_C(shortns['sourceC'])
cs=shortns['sourceC'];exec(cs[cs.index('def predict('):cs.index('def threshold(')],shortns['nc'][0])
shortns['nc'][0]['memo']={}

d=pd.read_csv(WORK/'reports/tables/original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
gas=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.eq('가스'))].reset_index(drop=True)
tail=n.cells[2].source[n.cells[2].source.index('na,nb,nd,nc='):]
tail=tail.replace('yy,[42,7],False','yy,[42],False').replace('for seeds in [[42,7],[11,23],[101,37]]:','for seeds in [[42]]:')
tail=tail.replace("processed=ROOT/'dataset/processed/rg3'","processed=WORK/'reports/tables/shortshot15_candidate_data'")
tail=tail.replace("for target,definition in list(zip([na,nb,nd],definitions))","definitions[1]=reduce_B(definitions[1])\nfor target,definition in list(zip([na,nb,nd],definitions))")
exec(tail)
loaded=joblib.load(MODEL_DIR/'shortshot_best.joblib')
p2,s2=predict_short(loaded,f)
assert np.array_equal(p,p2) and np.allclose(s,s2,atol=1e-12)
count=1+sum(len(ms) for ms in loaded['A']['modes'])+len(loaded['B']['models'])+sum(len(c['models']) for c in loaded['C'])+1
assert count==15
summary={'classifiers':count,'A':9,'B':2,'C':3,'D':1,'fullfit_score_matches_source':True,'save_reload_score_matches':True,'OOF_all591_predictions_match_original38':True,'OOF_TP':6,'OOF_FP':11,'OOF_FN':2,'OOF_TN':555,'OOF_F1':.48,'deployment_model_replaced':False,'pushed':False}
(MODEL_DIR/'verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(summary)


oof=pd.read_csv(WORK/'reports/tables/rg3_204_shortshot_reduction_predictions.csv')
candidate=oof[oof.candidate.eq('pruned15')]
original=oof[oof.candidate.eq('original38')]
matched=candidate.merge(original[['TimeStamp','prediction']],on='TimeStamp',suffixes=('_18','_38'),validate='one_to_one')
assert len(matched)==591 and matched.prediction_18.eq(matched.prediction_38).all()
print('15-classifier OOF decisions equal original38 all591:',True)



